# Resilience: Outcomes, Stalls, and Plain Logs


In [1]:
import io
import logging
import tempfile
import time
from pathlib import Path

import psnek

logger = logging.getLogger("demo")
workdir = Path(tempfile.mkdtemp(prefix="psnek-demo-"))

## Outcomes

Every finished bar has an `outcome`: `done`, `failed`, or `interrupted`. Each is drawn with its own glyph and word, not just a color.


In [4]:
keep = psnek.Options(leave=True)
with psnek.Bar(total=20, desc="completes", options=keep) as done_bar:
    for _ in range(20):
        time.sleep(0.01)
        done_bar.update()

try:
    with psnek.Bar(total=20, desc="raises", options=keep) as failed_bar:
        for step in range(20):
            time.sleep(0.01)
            if step == 12:
                raise ValueError("bad record")
            failed_bar.update()
except ValueError as error:
    logger.error("job failed: %s", error)

with psnek.Bar(total=20, desc="cancelled", options=keep) as cancelled_bar:
    for step in range(20):
        time.sleep(0.01)
        if step == 7:
            cancelled_bar.interrupt()
            break
        cancelled_bar.update()

print([bar.outcome for bar in (done_bar, failed_bar, cancelled_bar)])

# completes ##########################  100.0%      20/20         0:00   81.1it/s  done
x raises    ###############+__________   60.0%      12/20         0:00   75.5it/s  failed ValueError
! cancelled #########.________________   35.0%       7/20         0:00   74.6it/s  interrupted

job failed: bad record


['done', 'failed', 'interrupted']


## Stalls and Plain Status Lines

Without a terminal or notebook, such as in CI or when run by an AI agent, psnek prints plain status lines instead: a start line, one every 10 seconds, one whenever the bar's health changes, and a final summary.

Writing the bar to a `StringIO` uses that log mode here. The loop below hangs for 6 seconds midway, so psnek reports the stall, then the recovery.


In [5]:
log = io.StringIO()
with psnek.Bar(total=100, desc="ingest", options=psnek.Options(stream=log)) as bar:
    for step in range(100):
        time.sleep(6 if step == 50 else 0.02)
        bar.update()
        if step == 50:
            bar.panel({"waited_s": 6, "cause": "slow upstream"})
print(log.getvalue())

ingest: 0.0% 0/100 0:00<--:-- --it/s
ingest: 50.0% 50/100 0:06<~0:06 stalled 5s 40.3it/s
ingest: 51.0% 51/100 0:07<~0:06 7.1it/s (recovered)
waited_s  6
cause     slow upstream
ingest: 100.0% 100/100 0:08 done 14.0it/s



## Running Notebooks Headless

When a tool such as papermill or nbclient executes a notebook, set `PSNEK_MODE=log` to get these plain lines in the cell output instead.
